# Colab M1 runtime validation

This notebook remains a thin launcher. It mounts Google Drive, calls the project-owned bootstrap/doctor CLIs and displays their real JSON reports. It does not start training or manufacture outputs.

In [ ]:
import json
import os
import subprocess
from datetime import datetime, timezone
from pathlib import Path

from IPython.display import JSON, display
from google.colab import drive

drive.mount("/content/drive")
repo = Path.cwd()
data_root = Path("/content/drive/MyDrive/vla-fewshot")
scratch_root = Path("/content/vla_scratch")
os.environ.update({
    "VLA_PROJECT_ROOT": str(repo),
    "VLA_DATA_ROOT": str(data_root),
    "VLA_SCRATCH_DIR": str(scratch_root),
    "HF_HOME": str(data_root / "cache" / "huggingface"),
    "TORCH_HOME": str(data_root / "cache" / "torch"),
    "MUJOCO_GL": "egl",
    "TOKENIZERS_PARALLELISM": "false",
    "WANDB_MODE": "disabled",
    "WANDB_DISABLED": "true",
})

subprocess.run(["python", "-m", "pip", "install", "uv==0.6.13"], check=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
bootstrap_dir = data_root / "bootstrap" / f"colab-{stamp}"
doctor_dir = data_root / "doctor" / f"colab-{stamp}"
subprocess.run(
    ["bash", "scripts/bootstrap_colab.sh", "--output-dir", str(bootstrap_dir)],
    cwd=repo,
    check=True,
)
subprocess.run(
    [
        "uv", "run", "python", "scripts/doctor.py",
        "--config", "configs/platform/colab.yaml",
        "--profile", "full",
        "--output-dir", str(doctor_dir),
    ],
    cwd=repo,
    check=True,
)
display(JSON(json.loads((doctor_dir / "doctor.json").read_text())))